In [1]:
import time

import pandas as pd

from utilities.preprocessor import *
from utilities.values import *

In [2]:
data_dir = './data/'
dataset_preprocessor = DatasetPreprocessor()

### 1. Merge the separate raw datafiles

In [3]:
# combine all the raw files
raw_data_dir_paths = [data_dir + 'raw/2020', data_dir + 'raw/2021',]
full_dataset_df = dataset_preprocessor.read_all_files_in_dir(raw_data_dir_paths)

# sort by increasing start and end dates
full_dataset_df = full_dataset_df.sort_values(['starttime', 'stoptime'])
print('Total number of rows:', len(full_dataset_df))

Total number of rows: 393312


### 2. Preprocessing the dataset

#### 2a. Extract date features and get the rental counts for each station, date and time

In [4]:
# remove the depot, which only has 1 rental and is not an actual station
full_dataset_df = full_dataset_df[full_dataset_df[STATION] != 'JCBS Depot']

# force starttime into datetime format so that it is usable
full_dataset_df['starttime'] = pd.to_datetime(full_dataset_df['starttime'])

# extract hour first, so that we can make sure that each station-date-hour has a value
full_dataset_df[DATE] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_date)
full_dataset_df[HOUR_OF_DAY] = full_dataset_df['starttime'].apply(dataset_preprocessor.to_hour_of_day)

# count rentals for each station-date-hour (only includes those with at least 1 rental)
rental_counts_df = full_dataset_df.groupby([STATION, DATE, HOUR_OF_DAY]).size().reset_index(name = NUM_RENTALS)

In [5]:
# make sure that every station-date-hour has a row, even if 0
stations = rental_counts_df[STATION].unique()
dates = pd.date_range('2020-01-01', '2021-04-30', freq = 'D').date
hours = range(24)

full_rental_counts_df = pd.MultiIndex.from_product(
    [stations, dates, hours], names = [STATION, DATE, HOUR_OF_DAY]
).to_frame(index = False)

# merge the rental counts, fill rows with no values to 0
full_rental_counts_df = full_rental_counts_df.merge(rental_counts_df, on = [STATION, DATE, HOUR_OF_DAY], how = 'left')
full_rental_counts_df[NUM_RENTALS] = full_rental_counts_df[NUM_RENTALS].fillna(0).astype(int)

# merge the coordinates of each station
station_locations_df = full_dataset_df.groupby(STATION)[[LATITUDE, LONGITUDE]].first().reset_index()
full_rental_counts_df = full_rental_counts_df.merge(station_locations_df, on = STATION, how = 'left')

In [6]:
# extract the other date features
full_rental_counts_df[YEAR] = full_rental_counts_df[DATE].apply(dataset_preprocessor.to_year)
full_rental_counts_df[MONTH] = full_rental_counts_df[DATE].apply(dataset_preprocessor.to_month)
full_rental_counts_df[WEEK_OF_YEAR] = full_rental_counts_df[DATE].apply(dataset_preprocessor.to_week_of_year)
full_rental_counts_df[DAY_OF_WEEK] = full_rental_counts_df[DATE].apply(dataset_preprocessor.to_day_of_week)
full_rental_counts_df[IS_PUBLIC_HOLIDAY] = full_rental_counts_df[DATE].apply(dataset_preprocessor.to_is_public_holiday)

# sort for easy reading
full_rental_counts_df = full_rental_counts_df.sort_values([STATION, DATE, HOUR_OF_DAY])

In [7]:
full_rental_counts_df.head()

,start station name,date,hour_of_day,num_rentals,start station latitude,start station longitude,year,month,week_of_year,day_of_week,is_public_holiday
0,5 Corners Library,2020-01-01,0,0,40.734961,-74.059503,2020,1,1,2,1
1,5 Corners Library,2020-01-01,1,0,40.734961,-74.059503,2020,1,1,2,1
2,5 Corners Library,2020-01-01,2,0,40.734961,-74.059503,2020,1,1,2,1
3,5 Corners Library,2020-01-01,3,0,40.734961,-74.059503,2020,1,1,2,1
4,5 Corners Library,2020-01-01,4,0,40.734961,-74.059503,2020,1,1,2,1


#### 2b. Create new features using the actual rental counts of the last 3 hours

In [8]:
# timestamp of each row, so that T-1 works across midnight
rental_timestamp = (
    pd.to_datetime(full_rental_counts_df[DATE])
    + pd.to_timedelta(full_rental_counts_df[HOUR_OF_DAY], unit = 'h')
)

# lookup table: rentals for each station at each hour
rentals_lookup = (
    full_rental_counts_df
    .assign(timestamp = rental_timestamp)
    .groupby([STATION, 'timestamp'])[NUM_RENTALS]
    .sum()
)
data_start = rental_timestamp.min()

for lag in [1, 2, 3]:
    lag_timestamp = rental_timestamp - pd.Timedelta(hours = lag)
    lag_keys = pd.MultiIndex.from_arrays([full_rental_counts_df[STATION], lag_timestamp])

    # same station, exactly `lag` hours earlier; no row for that hour means 0 rentals
    lag_values = pd.Series(
        rentals_lookup.reindex(lag_keys).fillna(0).to_numpy(),
        index = full_rental_counts_df.index,
    )

    # hours before the dataset begins are unknown, not 0
    full_rental_counts_df[f'{NUM_RENTALS}_t-{lag}'] = lag_values.where(lag_timestamp >= data_start)

# save to file
full_rental_counts_df.to_csv(data_dir + 'intermediate_processing/full_rental_counts.csv', index=False)
print('Total number of rows:', len(full_rental_counts_df))

Total number of rows: 594864


In [9]:
full_rental_counts_df.head()

,start station name,date,hour_of_day,num_rentals,start station latitude,start station longitude,year,month,week_of_year,day_of_week,is_public_holiday,num_rentals_t-1,num_rentals_t-2,num_rentals_t-3
0,5 Corners Library,2020-01-01,0,0,40.734961,-74.059503,2020,1,1,2,1,NaN,NaN,NaN
1,5 Corners Library,2020-01-01,1,0,40.734961,-74.059503,2020,1,1,2,1,0.0,NaN,NaN
2,5 Corners Library,2020-01-01,2,0,40.734961,-74.059503,2020,1,1,2,1,0.0,0.0,NaN
3,5 Corners Library,2020-01-01,3,0,40.734961,-74.059503,2020,1,1,2,1,0.0,0.0,0.0
4,5 Corners Library,2020-01-01,4,0,40.734961,-74.059503,2020,1,1,2,1,0.0,0.0,0.0


#### 2c. Extract weather features

In [10]:
# get the location and dates to query
weather_search_df = full_rental_counts_df[[STATION, LATITUDE, LONGITUDE, DATE]]
weather_search_df = weather_search_df.drop_duplicates()
stations = set(weather_search_df[STATION])

In [11]:
# call the weather API for each location
weather_df = pd.DataFrame()

for i, station in enumerate(stations, start = 1):
    location_df = weather_search_df[weather_search_df[STATION] == station]
    location_df = location_df.sort_values(DATE)

    params = {
        START_DATE: location_df[DATE].iloc[0],
        END_DATE: location_df[DATE].iloc[-1],
        STATION: station,
        LATITUDE: location_df[LATITUDE].iloc[0],
        LONGITUDE: location_df[LONGITUDE].iloc[0]
    }

    # get historical weather data
    station_weather_history_df = dataset_preprocessor.get_hourly_weather(params)
    weather_df = pd.concat([weather_df, station_weather_history_df])

    # pause for 1 min after every 20 calls to stay under the API rate limit
    if i % 20 == 0:
        print(f"{i} stations done, sleeping for 60s...")
        time.sleep(60)

# save results
weather_df.to_csv(data_dir + 'intermediate_processing/historical_weather_features.csv', index=False)

20 stations done, sleeping for 60s...
40 stations done, sleeping for 60s...


In [12]:
weather_df.head()

,date,temperature_2m,rain,snow_depth,snowfall,cloud_cover_low,cloud_cover_mid,cloud_cover_high,relative_humidity_2m,precipitation,wind_speed_10m,wind_gusts_10m,cloud_cover,hour_of_day,start station name,start station latitude,start station longitude
0,2020-01-01,3.20,0.0,0.0,0.0,0.0,4.0,0.0,73.943901,0.0,17.673029,29.160000,4.0,0,Sip Ave,40.730897,-74.063913
1,2020-01-01,2.05,0.0,0.0,0.0,0.0,0.0,0.0,76.498848,0.0,16.559999,29.160000,0.0,1,Sip Ave,40.730897,-74.063913
2,2020-01-01,1.55,0.0,0.0,0.0,0.0,0.0,0.0,78.990501,0.0,16.563911,27.719999,0.0,2,Sip Ave,40.730897,-74.063913
3,2020-01-01,1.20,0.0,0.0,0.0,0.0,0.0,11.0,76.922134,0.0,18.723459,31.680000,11.0,3,Sip Ave,40.730897,-74.063913
4,2020-01-01,0.85,0.0,0.0,0.0,0.0,0.0,71.0,76.295395,0.0,17.339897,31.319998,71.0,4,Sip Ave,40.730897,-74.063913


#### 2d. Merge the citibike and weather features

In [13]:
# read previously stored intermediate files
full_rental_counts_df = pd.read_csv(data_dir + 'intermediate_processing/full_rental_counts.csv')
weather_df = pd.read_csv(data_dir + 'intermediate_processing/historical_weather_features.csv')

In [14]:
merge_columns = [STATION, DATE, HOUR_OF_DAY]

# one weather row per station-date-hour
# drops the repeated hour on 1 Nov daylight savings end
weather_to_merge_df = (
    weather_df
    .drop(columns = [LATITUDE, LONGITUDE])
    .drop_duplicates(subset = merge_columns, keep = "first")
)

# lag the weather by 1 hour: the weather recorded at hour T-1 is used to predict hour T
# done on a timestamp so that 11pm rolls over to 12am of the next date
lagged_timestamp = (
    pd.to_datetime(weather_to_merge_df[DATE])
    + pd.to_timedelta(weather_to_merge_df[HOUR_OF_DAY] + 1, unit = 'h')
)
weather_to_merge_df[DATE] = lagged_timestamp.dt.strftime('%Y-%m-%d')
weather_to_merge_df[HOUR_OF_DAY] = lagged_timestamp.dt.hour

# merge features
full_features_df = full_rental_counts_df.merge(
    weather_to_merge_df,
    on = merge_columns,
    how = "left",
    validate = "many_to_one",
)

# drop columns that will not be used
full_features_df = full_features_df.drop(columns=[LATITUDE, LONGITUDE])

# save results
full_features_df.to_csv(data_dir + 'intermediate_processing/full_features.csv', index=False)

#### 2e. Perform one-hot encoding and feature normalisation

#### 2f. Train-validation-test split